# MSC dense vs validated (single patient)

Compare dense and surrogate-validated MSC matrices using cached outputs.

**Legacy notebooks merged:**
- UTILS-COHERENCE_NETWORKS.ipynb

In [ ]:
%matplotlib inline
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")
from lrg_eegfc.notebook import *

In [ ]:
DATA_ROOT = Path('data/stereoeeg_patients')
patients = list_patients(DATA_ROOT)
assert patients, 'No patients found under data/stereoeeg_patients'

patient = patients[0]
phase = PHASE_LABELS[0]
band = BRAIN_BANDS_NAMES[2]

nperseg = 512
n_surrogates = 50
filter_time = 5000

msc_dense = load_msc_matrix(
    patient,
    phase,
    band,
    nperseg=nperseg,
    sparsify='none',
    n_surrogates=0,
)
if msc_dense is None:
    dense = compute_msc_matrix(
        patient,
        phase,
        band,
        nperseg=nperseg,
        sparsify='none',
        n_surrogates=0,
        filter_time=filter_time,
        cache_root=Path('data/msc_cache_dev'),
        verbose=True,
    )
    msc_dense = dense.adjacency_matrix

msc_soft = load_msc_matrix(
    patient,
    phase,
    band,
    nperseg=nperseg,
    sparsify='soft',
    n_surrogates=n_surrogates,
)
if msc_soft is None:
    soft = compute_msc_matrix(
        patient,
        phase,
        band,
        nperseg=nperseg,
        sparsify='soft',
        n_surrogates=n_surrogates,
        filter_time=filter_time,
        cache_root=Path('data/msc_cache_dev'),
        verbose=True,
    )
    msc_soft = soft.adjacency_matrix

msc_dense.shape, msc_soft.shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(msc_dense, cmap='viridis', vmin=0, vmax=1)
axes[0].set_title('MSC dense')
axes[1].imshow(msc_soft, cmap='viridis', vmin=0, vmax=1)
axes[1].set_title('MSC validated (soft)')
plt.tight_layout()
plt.show()

tri = np.triu_indices_from(msc_dense, k=1)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(msc_dense[tri], bins=60, alpha=0.6, label='dense')
ax.hist(msc_soft[tri], bins=60, alpha=0.6, label='validated')
ax.set_title('Upper-triangle distributions')
ax.legend()
plt.tight_layout()
plt.show()

print('dense mean', float(np.mean(msc_dense[tri])))
print('soft mean', float(np.mean(msc_soft[tri])))